# Data Acquisition, Merge, and Checks

Given the large dataset, several notebooks were used to originally concatenate the data from NYISO, pull data from ERA5, format and standardize the data, and conduct initial data checks. The code below is a simplified version of the code across notebooks from Google Colab, Kaggle, and Jupyter Notebook. I experimented with different tools to determine which would best handle this large dataset with the least amount of lag during the initial stage of data acquisition, merging, and checking. That is why this code is mostly in markdown.

In [ ]:
import os
import zipfile
from pathlib import Path

import pandas as pd

In [ ]:
repo_root = Path.home() / "Documents" / "Coding" / "solar_forecast"

data_root = repo_root / "data"
raw_dir = data_root / "raw"
interim_dir = data_root / "interim"
processed_dir = data_root / "processed"

solar_raw_dir = raw_dir / "nyiso_solar"

In [ ]:
nyiso_dirs = {
    "actuals": solar_raw_dir / "NYISO_Actuals",
    "forecasts": solar_raw_dir / "NYISO_Forecasts",
    "capacity": solar_raw_dir / "NYISO_Capacity",
}

unzipped_dirs = {
    k: raw_dir / f"unzipped_{k}" for k in nyiso_dirs
}

In [ ]:
nyiso_out = interim_dir / "01_nyiso_merged.csv"
era5_src = raw_dir / "era5_nyiso_center_2020_2025.csv"
era5_out = interim_dir / "02_era5_weather.csv"
merged_out = processed_dir / "03_merged_data.csv"

interim_dir.mkdir(parents=True, exist_ok=True)
processed_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
# Creates a function that unzips all files in sf and extracts the zip file, only issue is it might overwrite some zip files if they have the same name and doesn't do a good job of validating bad or empty content, might recurse and use a different method later.

def unzip_all_archives(input_folder, output_folder):
    os.makedirs(output_folder, exist_ok=True)
    extracted_files = 0

    for filename in os.listdir(input_folder):
        if filename.endswith(".zip"):
            try:
                with zipfile.ZipFile(os.path.join(input_folder, filename), "r") as archive:
                    archive.extractall(output_folder)
                    extracted_files += 1
            except Exception as e:
                print(f"Did Not Extract: {filename} | {e}")

    print(f"Extraction Completed: {extracted_files} archives from {input_folder}")

for key in nyiso_dirs:
    unzip_all_archives(nyiso_dirs[key], unzipped_dirs[key])

In [ ]:
def load_folder(folder: Path) -> pd.DataFrame:
    csv_files = list(folder.glob("*.csv"))
    frames = []

    for file in csv_files:
        try:
            df = pd.read_csv(file)
            df["source_file"] = file.name
            frames.append(df)
        except Exception as e:
            print(f"FAILED TO READ: {file.name} | {e}")

    if not frames:
        return pd.DataFrame()

    df_all = pd.concat(frames, ignore_index=True)
    return df_all

In [ ]:
df_actual = load_folder(unzipped_dirs["actuals"])
df_forecast = load_folder(unzipped_dirs["forecasts"])
df_capacity = load_folder(unzipped_dirs["capacity"])

In [ ]:
print("Loaded Shapes")
print(". . .")
print("Actuals:", df_actual.shape)
print("Forecasts:", df_forecast.shape)
print("Capacity:", df_capacity.shape)

In [ ]:
for df in (df_actual, df_forecast, df_capacity):
    df.columns = (
        df.columns.str.strip()
        .str.lower()
        .str.replace(" ", "_")
        .str.replace("-", "_")
    )

ts_col = "time_stamp"
zone_col = "zone_name"
val_col = "mw_value"

In [ ]:
for df in (df_actual, df_forecast, df_capacity):
    df[ts_col] = pd.to_datetime(df[ts_col], errors="coerce")
    df[ts_col] = df[ts_col].dt.tz_localize(
        "America/New_York",
        nonexistent="shift_forward",
        ambiguous="NaT",
    )
    df[ts_col] = df[ts_col].dt.tz_convert("UTC")
    df[ts_col] = df[ts_col].dt.floor("H")

for df in (df_actual, df_forecast, df_capacity):
    df[zone_col] = df[zone_col].astype(str).str.upper().str.strip()

for df in (df_actual, df_forecast, df_capacity):
    df[zone_col] = df[zone_col].str.upper().str.strip()

In [ ]:
df_nyiso = (
    df_actual.merge(
        df_forecast,
        how="outer",
        on=[ts_col, zone_col],
        suffixes=("_actual", "_forecast"),
    )
    .merge(
        df_capacity,
        how="outer",
        on=[ts_col, zone_col],
        suffixes=("", "_capacity"),
    )
)

df_nyiso.to_csv(nyiso_out, index=False)

In [ ]:
print("Final Shape:", df_nyiso.shape)
print("Zones Detected:", df_nyiso[zone_col].nunique())
print("Time Range:", df_nyiso[ts_col].min(), "→", df_nyiso[ts_col].max())
print("Saved:", nyiso_out)

In [ ]:
df_era5 = pd.read_csv(era5_src)
print(df_era5.columns)

In [ ]:
df_era5[ts_col] = pd.to_datetime(df_era5["time"], utc=True)
df_era5.to_csv(era5_out, index=False)

print("ERA5 Shape:", df_era5.shape)
print("ERA5 Time Range:", df_era5[ts_col].min(), "→", df_era5[ts_col].max())
print("Saved:", era5_out)

In [ ]:
df_merge = pd.merge(df_nyiso, df_era5, on=ts_col, how="inner")
df_merge.to_csv(merged_out, index=False)

print(df_merge.head())

In [ ]:
print("Merged Shape:", df_merge.shape)
print("Merged Time Range:", df_merge[ts_col].min(), "→", df_merge[ts_col].max())
print("Saved:", merged_out)

In [ ]:
print("Columns Check")
print(". . .")
print("NYISO Columns:", df_nyiso.columns.tolist())
print("ERA5 Columns:", df_era5.columns.tolist())
print("Merged Columns:", df_merge.columns.tolist())

In [ ]:
print("Missing Check")
print(". . .")
print("NYISO Missing time_stamp:", df_nyiso[ts_col].isna().sum())
print("ERA5 Missing time_stamp:", df_era5[ts_col].isna().sum())
print("Merged Missing time_stamp:", df_merge[ts_col].isna().sum())

In [ ]:
print("Duplication Check")
print(". . .")
print("NYISO Duplicates:", df_nyiso.duplicated(subset=[ts_col, zone_col]).sum())
print("Merged Duplicates:", df_merge.duplicated(subset=[ts_col, zone_col]).sum() if zone_col in df_merge.columns else "zone_name not found")